[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ach0000/memorization_generalization_in_diffusion_models_tests/blob/main/tp/TP_gaussien_wiener.ipynb)

# Le cas gaussien : filtre de Wiener et base de Fourier

Si les images suivaient une loi gaussienne, le débruiteur optimal serait linéaire (filtre de Wiener) : il réduirait les coefficients de l'image dans une base **fixe**, qui ne dépend pas de l'image observée. Pour une loi stationnaire, cette base est celle de Fourier. On met ce débruiteur à l'épreuve sur des images qui, elles, ne sont pas gaussiennes.

**Matériel** : tourne sur un ordinateur personnel (CPU) en quelques minutes ; sur Colab, exécuter d'abord la cellule d'installation (GPU facultatif).

In [ ]:
# Installation (Colab) : à exécuter en premier ; en local, ne fait que rendre le code importable
GITHUB_USER = "ach0000"  # propriétaire du fork
REPO = "memorization_generalization_in_diffusion_models_tests"
BRANCH = "main"
# seuls ces fichiers du dépôt sont téléchargés (motifs de git sparse-checkout)
FICHIERS = ['/code/',
            '/tp/tp_outils.py',
            '/tp/prepare_celeba80x80.py',
            '/denoisers/UNet/C_alpha4_beta4_line_80x80_blurredx2/']

import os, sys, subprocess
if "google.colab" in sys.modules:
    racine = os.path.join("/content", REPO)
    if not os.path.isdir(os.path.join(racine, ".git")):
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "-b", BRANCH,
                        "https://github.com/%s/%s.git" % (GITHUB_USER, REPO), racine], check=True)
    subprocess.run(["git", "-C", racine, "sparse-checkout", "set", "--no-cone", *FICHIERS], check=True)
    subprocess.run(["git", "-C", racine, "checkout", BRANCH], check=True)
    try:
        import pywt   # PyWavelets (générateur d'images C^alpha des auteurs) : absent de certaines images Colab
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PyWavelets"], check=True)
else:
    racine = os.getcwd()
    while not os.path.exists(os.path.join(racine, "code", "main.py")) and os.path.dirname(racine) != racine:
        racine = os.path.dirname(racine)
os.makedirs(os.path.join(racine, "tp"), exist_ok=True)   # absent du clone partiel
os.chdir(os.path.join(racine, "tp"))
for d in ("code", "tp"):
    if os.path.join(racine, d) not in sys.path:
        sys.path.insert(0, os.path.join(racine, d))
print("Répertoire courant :", os.getcwd())

## 0. Réglages

In [ ]:
%matplotlib inline
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import tp_outils as T
T.nommer_figures("TP_gaussien_wiener")   # figures nommées dans resultats/figures/

SOURCE = "calpha"   # ou "celeba" (téléchargement d'environ 1,4 Go : à réserver à Colab)
GRAINE = 0
ALPHA = 4
K = 80

## 1. Les données

20 000 images pour estimer la moyenne et le spectre, 200 images de test distinctes.

In [ ]:
if SOURCE == "calpha":
    X = T.images_calpha(ALPHA, 20000, K, graine=GRAINE)
    X_test = T.images_calpha(ALPHA, 200, K, graine=GRAINE + 1)
else:
    tout = T.charger_celeba()
    X, X_test = tout[:20000], tout[-200:]
d = K * K
print(tuple(X.shape), tuple(X_test.shape))

## 2. Modèle gaussien stationnaire

Une loi est **stationnaire** si ses statistiques ne changent pas par translation : sa moyenne est constante
et la covariance entre deux pixels ne dépend que de leur décalage. Sa matrice de covariance est alors
(aux bords près, que l'on traite comme périodiques) diagonalisée par la transformée de Fourier discrète :
les vecteurs propres sont les sinusoïdes $e^{2i\pi\langle \omega, n\rangle/K}$ et les valeurs propres forment le
**spectre de puissance** $P(\omega) = \mathbb E\,|\hat x(\omega) - \hat\mu(\omega)|^2$ (transformée orthonormée).

On estime la moyenne (un seul nombre, par stationnarité) et le spectre (moyenne des périodogrammes). Le spectre
donne la variance de chaque coefficient de Fourier ; triées par ordre décroissant, ces variances $P_{(k)}$
décroissent avec le rang $k$. Une décroissance en $k^{-(\alpha_F + 1)}$ apparaît comme une droite en échelle
log-log (section 3.1 de l'énoncé).

In [ ]:
mu = X.mean().item()
F = torch.fft.fft2(X[:, 0] - mu, norm="ortho")
P = (F.abs() ** 2).mean(dim=0)                       # spectre de puissance estimé (K, K)
P_tries = np.sort(P.numpy().ravel())[::-1]
rangs = np.arange(1, d + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im = axes[0].imshow(np.fft.fftshift(np.log10(P.numpy())), cmap="viridis",
                    extent=[-K / 2, K / 2, K / 2, -K / 2])
fig.colorbar(im, ax=axes[0], label="log10 P(ω)")
axes[0].set_title("Spectre de puissance (fréquence nulle au centre)")
axes[0].set_xlabel("ω_x"); axes[0].set_ylabel("ω_y")
axes[1].loglog(rangs, P_tries, lw=1.5)
axes[1].set_xlabel("rang k"); axes[1].set_ylabel("variance P_(k) du k-ième coefficient")
axes[1].set_title("Variances des coefficients de Fourier, triées"); axes[1].grid(alpha=0.3)
T.montrer("spectre")
print("moyenne %.3f ; 10 plus grandes variances :" % mu, np.round(P_tries[:10], 2))

### Vérification sans hypothèse de stationnarité : l'ACP

Sans supposer la stationnarité, la covariance empirique se diagonalise par ACP. On le fait sur des images
réduites à 32×32 ($d = 1\,024$, 20 000 images) : si la loi est presque stationnaire, les premiers vecteurs
propres ressemblent à des sinusoïdes.

In [ ]:
X32 = torch.nn.functional.interpolate(X, size=(32, 32), mode="area")   # réduction par moyenne locale
Z = X32.flatten(1) - X32.flatten(1).mean(dim=0)
cov = Z.T @ Z / (len(Z) - 1)
valeurs_acp, vecteurs_acp = torch.linalg.eigh(cov)
valeurs_acp, vecteurs_acp = valeurs_acp.flip(0), vecteurs_acp.flip(1)
T.afficher_vecteurs(vecteurs_acp[:, :14].T, 32, ["e_%d" % k for k in range(1, 15)], ncol=7,
                    titre="Premiers vecteurs propres de la covariance (ACP, 32×32)", nom="acp")

## 3. Le débruiteur optimal gaussien : le filtre de Wiener

Pour $x \sim \mathcal N(\mu, \Sigma)$ et $y = x + \sigma z$, la loi a posteriori est gaussienne et

$$f^\star(y) = \mathbb E[x\mid y] = \mu + \Sigma(\Sigma + \sigma^2 I)^{-1}(y - \mu),\qquad
\nabla f^\star(y) = \Sigma(\Sigma + \sigma^2 I)^{-1}\ \text{(constant)}.$$

Dans la base de Fourier, chaque coefficient est multiplié par $\lambda(\omega) = P(\omega) / (P(\omega) + \sigma^2)$ :
c'est un rétrécissement dans une base **fixe**, avec des facteurs qui ne dépendent pas de $y$.

In [ ]:
def wiener(y, sigma):
    Y = torch.fft.fft2(y[:, 0] - mu, norm="ortho")
    lam = P.to(y.device) / (P.to(y.device) + sigma ** 2)
    return (mu + torch.fft.ifft2(lam * Y, norm="ortho").real).unsqueeze(1)

fig, ax = plt.subplots(figsize=(6, 4))
for sigma in [10 / 255, 50 / 255, 150 / 255]:
    lam = (P / (P + sigma ** 2)).numpy()
    ax.semilogy(np.sort(lam.ravel())[::-1], label="σ = %.2f : tr J = %.0f" % (sigma, lam.sum()))
ax.set_xlabel("rang k (coefficients triés)"); ax.set_ylabel("λ_k"); ax.set_ylim(1e-3, 1.1)
ax.grid(alpha=0.3); ax.legend(); ax.set_title("Facteurs de rétrécissement du filtre de Wiener")
T.montrer("retrecissement")

chemin_reseau = "../denoisers/UNet/C_alpha4_beta4_line_80x80_blurredx2"
reseau = None
if SOURCE == "calpha" and ALPHA == 4 and os.path.isdir(chemin_reseau):
    reseau = T.charger_unet_auteurs("C_alpha4_beta4_line_80x80_blurredx2", 100000)

torch.manual_seed(GRAINE)
x = X_test[:1]; sigma = 100 / 255
y = x + sigma * torch.randn_like(x)
images, titres = [x, y, wiener(y, sigma)], ["propre", "bruitée σ = %.2f" % sigma, "Wiener"]
if reseau is not None:
    with torch.no_grad():
        images.append((y.to(T.DEVICE) - reseau(y.to(T.DEVICE))).cpu()); titres.append("réseau (N = 10^5)")
titres = [t + "\n%.1f dB" % T.psnr(x, im.cpu()).item() if i else t for i, (t, im) in enumerate(zip(titres, images))]
T.afficher_images(torch.cat(images), titres, ncol=len(images), taille=2.5, nom="exemple")

## 4. Performances : courbes PSNR et pentes

On compare le filtre de Wiener et, s'il est disponible, le réseau des auteurs ($10^5$ images $C^4$). Les
pentes sont estimées sur les PSNR d'entrée supérieurs à 15 dB (petits bruits).

In [ ]:
sig_range = np.logspace(np.log10(5), np.log10(255), 12) / 255
courbes = {"Wiener (gaussien stationnaire)": T.courbe_psnr(wiener, X_test, sig_range)}
if reseau is not None:
    courbes["réseau des auteurs"] = T.courbe_psnr(lambda y, s: y - reseau(y), X_test, sig_range)
ax = T.trace_psnr(courbes, sig_range, "Images de test")
T.montrer("psnr")
x_in = T.psnr_entree(sig_range)
for nom, valeurs in courbes.items():
    print("%-32s pente %.2f   PSNR : %s" % (nom, T.pente(x_in, valeurs, 15), np.round(valeurs, 1)))

# décroissance des variances de Fourier sur les rangs où P_(k) est comparable aux sigma^2 de l'ajustement
s_fit = sig_range[x_in >= 15]
g = (P_tries <= s_fit.max() ** 2) & (P_tries >= s_fit.min() ** 2)
alpha_F = -np.polyfit(np.log(rangs[g]), np.log(P_tries[g]), 1)[0] - 1
print("rangs k de %d à %d : P_(k) ~ k^-(α_F+1) avec α_F = %.2f" % (rangs[g].min(), rangs[g].max(), alpha_F))
print("pente prédite pour Wiener α_F/(α_F+1) = %.2f ; pente optimale (base adaptée) α/(α+1) = %.2f"
      % (alpha_F / (alpha_F + 1), ALPHA / (ALPHA + 1)))

## 5. Ce que « gaussien » veut dire : échantillons du modèle

Un échantillon du modèle gaussien stationnaire s'obtient en colorant un bruit blanc :
$x = \mu + \mathcal F^{-1}\big(\sqrt{P}\,\mathcal F(w)\big)$, $w \sim \mathcal N(0, I)$. Il a la même moyenne et le même
spectre (donc la même covariance) que les images.

In [ ]:
g = torch.Generator().manual_seed(GRAINE)
w = torch.randn(10, K, K, generator=g)
X_gauss = (mu + torch.fft.ifft2(P.sqrt() * torch.fft.fft2(w, norm="ortho"), norm="ortho").real).unsqueeze(1)
T.afficher_images(torch.cat([X_test[:10], X_gauss]), ncol=10, vmin=-0.2, vmax=1.2,
                  titre="en haut : images de test ; en bas : échantillons du modèle gaussien", nom="echantillons")

### L'erreur du filtre de Wiener ne dépend que du spectre

Pour le filtre de Wiener, $\sigma^2\,\mathrm{tr}\,\nabla f = \sum_\omega \sigma^2 P(\omega)/(P(\omega) + \sigma^2)$. On
compare cette valeur à l'erreur quadratique mesurée sur les échantillons gaussiens et sur les vraies images.

In [ ]:
X_gauss = (mu + torch.fft.ifft2(P.sqrt() * torch.fft.fft2(torch.randn(2000, K, K, generator=g), norm="ortho"),
                                norm="ortho").real).unsqueeze(1)
torch.manual_seed(GRAINE)
for sigma in [10 / 255, 50 / 255, 150 / 255]:
    prediction = (sigma ** 2 * P / (P + sigma ** 2)).sum().item()
    mesures = []
    for donnees in [X_gauss, X[:2000]]:
        y = donnees + sigma * torch.randn_like(donnees)
        mesures.append(((wiener(y, sigma) - donnees) ** 2).flatten(1).sum(dim=1).mean().item())
    print("σ = %.3f : σ² tr J = %7.2f   erreur mesurée : gaussiennes %7.2f, images %7.2f" %
          (sigma, prediction, *mesures))